<a href="https://colab.research.google.com/github/ble27/CSCE-491/blob/main/FP32_INT8_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring FP32 to INT8 Quantization


In [1]:
# Install model/export dependencies without replacing Colab's CUDA-enabled PyTorch.
%pip install -q timm transformers onnx onnxscript onnxruntime datasets


## Shared packages for TensorRT and non-TensorRT pipeline


In [2]:
# Install ONNX PTQ and ModelOpt dependencies. Leave torch/numpy/protobuf versions to Colab.
%pip install -q -U "setuptools<82" wheel jedi
%pip install -q -U "optimum-onnx[onnxruntime]" "nvidia-modelopt[onnx]" onnxsim cuda-python


### Non-TensorRT download


In [3]:
# ONNX Runtime comparison uses the CPU provider for broad Colab compatibility.
# This is already installed above; keep this cell as a provider check.
import onnxruntime as ort
print("ORT providers:", ort.get_available_providers())


ORT providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


### TensortRT download

In [4]:
# TensorRT is needed only for the TensorRT section. Install its full builder/runtime package.
# NVIDIA's `tensorrt` metapackage selects the CUDA wheel supported by the current release.
%pip install -q -U tensorrt
import tensorrt as trt
print("TensorRT:", trt.__version__)


TensorRT: 11.3.0.99


In [5]:
# Runtime preflight. The ONNX Runtime pipeline can run on CPU, but ModelOpt
# and TensorRT require a real NVIDIA GPU runtime.
import shutil
import subprocess
import sys


def check_gpu_runtime():
    nvidia_smi = shutil.which("nvidia-smi")
    cuda_available = False
    cuda_version = None
    device_name = None

    try:
        import torch
        cuda_available = torch.cuda.is_available()
        cuda_version = torch.version.cuda
        if cuda_available:
            device_name = torch.cuda.get_device_name(0)
    except Exception as error:
        print(f"PyTorch CUDA check failed: {error}")

    print("Python:", sys.version.split()[0])
    print("nvidia-smi:", nvidia_smi or "not found")
    print("CUDA available:", cuda_available)
    print("CUDA version:", cuda_version)
    print("GPU:", device_name or "not available")

    if nvidia_smi:
        subprocess.run([nvidia_smi, "--query-gpu=name", "--format=csv,noheader"], check=False)

    return bool(nvidia_smi and cuda_available)


GPU_RUNTIME_AVAILABLE = check_gpu_runtime()


def require_gpu_runtime():
    if not GPU_RUNTIME_AVAILABLE:
        raise RuntimeError(
            "TensorRT/ModelOpt requires a Colab GPU runtime. Select "
            "Runtime > Change runtime type > T4 GPU, restart the runtime, "
            "and rerun the notebook from the beginning."
        )


print("GPU runtime ready:", GPU_RUNTIME_AVAILABLE)

Python: 3.13.15
nvidia-smi: /opt/bin/nvidia-smi
CUDA available: True
CUDA version: 13.0
GPU: Tesla T4
GPU runtime ready: True


In [6]:
# Non-TensorRT pipeline:
#  - FP32 HF model
#  - Export to ONNX
#  - ONNX runtime PTQ INT8 quantization
# - Benchmark / results

In [7]:
import sys
from pathlib import Path

import onnx
import onnxruntime as ort
import onnxscript
import timm
import torch
import transformers

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("ORT:", ort.__version__)
print("timm:", timm.__version__)
print("Transformers:", transformers.__version__)
print("ONNX Script:", onnxscript.__version__)
print("CUDA:", torch.cuda.is_available())
print("ORT providers:", ort.get_available_providers())

# Mount Drive only when it is not already mounted.
from google.colab import drive

DRIVE_ROOT = Path("/content/drive")
if not (DRIVE_ROOT / "MyDrive").exists():
    drive.mount(str(DRIVE_ROOT))

# Model choice: ResNet-18 is Conv+BN+ReLU only (like the paper's U-Net/SegResNet), so it
# quantizes cleanly with PTQ. MobileNetV3-small (HardSwish + SE blocks, tiny channels) does not.
PROJECT = DRIVE_ROOT / "MyDrive/PTQ_ResNet18"

for folder in [
    PROJECT / "models" / "fp32_onnx",
    PROJECT / "models" / "int8_onnx",
    PROJECT / "models" / "modelopt_qdq",
    PROJECT / "models" / "trt_engine",
    PROJECT / "data" / "calibration",
    PROJECT / "data" / "evaluation",
    PROJECT / "results",
]:
    folder.mkdir(parents=True, exist_ok=True)

FP32_ONNX = PROJECT / "models/fp32_onnx/resnet18_fp32.onnx"
INT8_ONNX = PROJECT / "models/int8_onnx/resnet18_ort_int8.onnx"

print("Project:", PROJECT)
print("FP32 model:", FP32_ONNX)

Python: 3.13.15
Torch: 2.11.0+cu130
ONNX: 1.21.0
ORT: 1.24.4
timm: 1.0.29
Transformers: 4.57.6
ONNX Script: 0.7.2
CUDA: True
ORT providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Project: /content/drive/MyDrive/PTQ_ResNet18
FP32 model: /content/drive/MyDrive/PTQ_ResNet18/models/fp32_onnx/resnet18_fp32.onnx


## Export PyTorch model to ONNX

In [8]:
# Non-TensorRT pipeline work
model_name = "resnet18.a1_in1k"
model = timm.create_model(model_name, pretrained=True)
model.eval()

# Find expected model input
config = timm.data.resolve_model_data_config(model)
print(config)

# PyTorch image models expect (batch_size, channels, height, width)
dummy_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)

# Export the model to ONNX
torch.onnx.export(
    model,
    (dummy_input,),
    str(FP32_ONNX),
    input_names=["image"],
    output_names=["logits"],
    opset_version=18,
    dynamo=True,
    verify=True,
)

assert FP32_ONNX.exists()
print(f"Saved successfully: {FP32_ONNX}")
print(f"Size: {FP32_ONNX.stat().st_size / 1e6:.2f} MB")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.95, 'crop_mode': 'center'}
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
[torch.onnx] Check the ONNX model...
[torch.onnx] Check the ONNX model... ✅
[torch.onnx] Execute the model with ONNX Runtime...
[torch.onnx] Execute the model with ONNX Runtime... ✅
[torch.onnx] Verify output accuracy...
[torch.onnx] Verify output accuracy... ✅
Saved successfully: /content/drive/MyDrive/PTQ_ResNet18/models/fp32_onnx/resnet18_fp32.onnx
Size: 0.09 MB


In [9]:
# Static Quantization: both weights and activation layers are pre-computed before quantization
# Use cases: best for image / vision models
# Pros: faster inference, higher throughput
# Cons: more setup, require calibration dataset since activation layers
# How: pass calibrated inputs to observe activation layers. Using activation layer, find X_min and X_max.
# From this calculate 2 values: Scale and Zero-Point
# These values are directly passed to calculate the Quantized Value during the quantization process without calculating dyamically


## Static Quantization pipeline for Non-TensorRT model


In [10]:
import sys

# ModelOpt monkey-patches ONNX Runtime's calibrator/quantizer when imported. If the
# ModelOpt cells already ran in this session, plain ORT quantization breaks with
# confusing errors ("Quantization parameters are not specified for param ...",
# "NoneType is not iterable"). Run this section BEFORE the ModelOpt cells, or restart.
if any(m.startswith("modelopt") for m in sys.modules):
    raise RuntimeError(
        "ModelOpt is already imported and has patched ONNX Runtime. "
        "Restart the runtime and run this cell before any ModelOpt cell."
    )

# Static quantization pipeline
import numpy as np
import onnxruntime
from datasets import load_dataset
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)

FP32_ONNX = PROJECT / "models/fp32_onnx/resnet18_fp32.onnx"
INT8_ONNX = PROJECT / "models/int8_onnx/resnet18_ort_int8.onnx"
INT8_ONNX.parent.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "resnet18.a1_in1k"
model = timm.create_model(MODEL_NAME, pretrained=True).eval()
calib_ds = load_dataset("AI-Lab-Makerere/beans", split="train").shuffle(seed=0).select(range(100))

data_cfg = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_cfg, is_training=False)

fp32_session = onnxruntime.InferenceSession(
    str(FP32_ONNX),
    providers=["CPUExecutionProvider"],
)
input_name = fp32_session.get_inputs()[0].name
print("ONNX input:", input_name)
print("ONNX shape:", fp32_session.get_inputs()[0].shape)
print("ONNX type:", fp32_session.get_inputs()[0].type)


class CalibrationReader(CalibrationDataReader):
    def __init__(self, dataset, input_name, transform):
        self.dataset = dataset
        self.input_name = input_name
        self.transform = transform
        self.index = 0

    def get_next(self):
        if self.index >= len(self.dataset):
            return None

        image = self.dataset[self.index]["image"].convert("RGB")
        tensor = self.transform(image).unsqueeze(0)
        self.index += 1
        return {self.input_name: tensor.numpy().astype(np.float32)}


# QDQ is ONNX Runtime's recommended static-quantization format. The legacy
# QOperator Conv rewriter fails on this Dynamo-exported graph (None.extend).
# Pre-process (shape inference, constant folding) to clean up the Dynamo-exported graph.
import onnx
from onnxruntime.quantization.shape_inference import quant_pre_process

PREPROCESSED_ONNX = FP32_ONNX.with_name("resnet18_fp32_preprocessed.onnx")
try:
    quant_pre_process(str(FP32_ONNX), str(PREPROCESSED_ONNX), skip_symbolic_shape=True)
except Exception as e:
    print("quant_pre_process failed, falling back to plain shape inference:", e)
    m = onnx.load(str(FP32_ONNX))  # also loads external data if present
    m = onnx.shape_inference.infer_shapes(m)
    onnx.save(m, str(PREPROCESSED_ONNX))
QUANT_INPUT = PREPROCESSED_ONNX

# Sanity check: the graph input must have type info and be consumed by a node.
_m = onnx.load(str(QUANT_INPUT))
print("Graph inputs:", [i.name for i in _m.graph.input])
print("Consumers of input:", [(n.op_type, n.name) for n in _m.graph.node if input_name in n.input])
assert [i.name for i in _m.graph.input] == [input_name], "input name mismatch with calibration reader"

try:
    quantize_static(
        model_input=str(QUANT_INPUT),
        model_output=str(INT8_ONNX),
        calibration_data_reader=CalibrationReader(
            calib_ds,
            input_name,
            transform,
        ),
        quant_format=QuantFormat.QDQ,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QUInt8,
        per_channel=True,
        calibrate_method=CalibrationMethod.MinMax,
    )
except ValueError as e:
    print("FULL ERROR:", e)  # the notebook traceback truncates this message
    raise

print("Saved INT8 model:", INT8_ONNX)

ONNX input: image
ONNX shape: [1, 3, 224, 224]
ONNX type: tensor(float)
Graph inputs: ['image']
Consumers of input: [('Conv', 'node_Conv_291')]
Saved INT8 model: /content/drive/MyDrive/PTQ_ResNet18/models/int8_onnx/resnet18_ort_int8.onnx


# TensorRT pipeline


In [11]:
"""
TensorRT pipeline:
  - FP32 HF Model
  - Export to ONNX
  - TensorRT calibration + INT8 engine build
  - Benchmark / results
"""

'\nTensorRT pipeline:\n  - FP32 HF Model\n  - Export to ONNX\n  - TensorRT calibration + INT8 engine build\n  - Benchmark / results\n'

In [12]:
from pathlib import Path
import numpy as np
import timm
from datasets import load_dataset

# Same as non-TensorRT for setup
PROJECT = Path("/content/drive/MyDrive/PTQ_ResNet18")
CALIB_NPY = PROJECT / "data/calibration/beans_calib_100.npy"
CALIB_NPY.parent.mkdir(parents=True, exist_ok=True)

model = timm.create_model(
    "resnet18.a1_in1k",
    pretrained=True,
).eval()

cfg = timm.data.resolve_model_data_config(model) # Determine the input size of the config
transform = timm.data.create_transform(**cfg, is_training=False)  # Transform function to scale the inputs

beans = load_dataset("AI-Lab-Makerere/beans", split="train").shuffle(seed=0).select(range(100))

calib = np.stack([
    transform(row["image"].convert("RGB")).numpy().astype(np.float32)
    for row in beans
]) # Stack each row of the beans dataset row-wise

np.save(CALIB_NPY, calib) # Save calibration dataset to CALIB_NPY path

print("Calibration shape:", calib.shape)
print("Saved:", CALIB_NPY)

Calibration shape: (100, 3, 224, 224)
Saved: /content/drive/MyDrive/PTQ_ResNet18/data/calibration/beans_calib_100.npy


## Fake ModelOpt INT8 QDQ




In [13]:
# ModelOpt creates an INT8 Q/DQ ONNX graph for TensorRT.
require_gpu_runtime()

import numpy as np
from modelopt.onnx.quantization import quantize

FP32_ONNX = PROJECT / "models/fp32_onnx/resnet18_fp32.onnx"
QDQ_ONNX = PROJECT / "models/modelopt_qdq/resnet18_int8_qdq.onnx"
QDQ_ONNX.parent.mkdir(parents=True, exist_ok=True)

calibration_data = np.load(CALIB_NPY)
calibration_input_name = onnxruntime.InferenceSession(
    str(FP32_ONNX), providers=["CPUExecutionProvider"]
).get_inputs()[0].name

quantize(
    onnx_path=str(FP32_ONNX),
    quantize_mode="int8",
    calibration_method="max",        # paper setting (default is entropy)
    high_precision_dtype="fp32",     # paper setting (default writes FP16 into the graph)
    calibration_data={calibration_input_name: calibration_data},
    output_path=str(QDQ_ONNX),
)
print("Saved QDQ ONNX:", QDQ_ONNX)

/usr/local/lib/python3.13/dist-packages/cupy/_environment.py:677: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda12x, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


2026-10-02 03:35:24,912 - [modelopt][onnx] - INFO - Starting quantization process for model: /content/drive/MyDrive/PTQ_ResNet18/models/fp32_onnx/resnet18_fp32.onnx


INFO:modelopt.onnx:Starting quantization process for model: /content/drive/MyDrive/PTQ_ResNet18/models/fp32_onnx/resnet18_fp32.onnx


2026-10-02 03:35:24,915 - [modelopt][onnx] - INFO - Quantization mode: int8


INFO:modelopt.onnx:Quantization mode: int8


...

2026-10-02 03:35:30,215 - [modelopt][onnx] - INFO - Validating quantized model


INFO:modelopt.onnx:Validating quantized model


2026-10-02 03:35:30,932 - [modelopt][onnx] - INFO - Quantization process completed


INFO:modelopt.onnx:Quantization process completed


Saved QDQ ONNX: /content/drive/MyDrive/PTQ_ResNet18/models/modelopt_qdq/resnet18_int8_qdq.onnx


### FP32 TensorRT Engine


In [14]:
# Build two separate engines for an honest comparison. FP32 is the validated
# baseline; INT8 is an experimental Q/DQ engine whose quality is measured below.
require_gpu_runtime()

import tensorrt as trt

FP32_ENGINE_PATH = PROJECT / "models/trt_engine/resnet18_fp32.engine"
INT8_ENGINE_PATH = PROJECT / "models/trt_engine/resnet18_int8_qdq.engine"
FP32_ENGINE_PATH.parent.mkdir(parents=True, exist_ok=True)

def build_trt_engine(onnx_path, engine_path, strongly_typed):
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    flags = (1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED)
             if strongly_typed else 0)
    network = builder.create_network(flags)
    parser = trt.OnnxParser(network, logger)

    if not parser.parse_from_file(str(onnx_path)):
        errors = [str(parser.get_error(i)) for i in range(parser.num_errors)]
        raise RuntimeError(f"TensorRT could not parse {onnx_path}: {errors}")

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
    # TensorRT enables TF32 by default. Disable it for a true FP32 baseline.
    config.clear_flag(trt.BuilderFlag.TF32)
    serialized = builder.build_serialized_network(network, config)
    if serialized is None:
        raise RuntimeError(f"TensorRT engine build failed for {onnx_path}")

    engine_path.write_bytes(bytes(serialized))
    print("Saved TensorRT engine:", engine_path)


build_trt_engine(FP32_ONNX, FP32_ENGINE_PATH, strongly_typed=False)
# Q/DQ is explicit quantization, so TensorRT must preserve its tensor types.
build_trt_engine(QDQ_ONNX, INT8_ENGINE_PATH, strongly_typed=True)

Saved TensorRT engine: /content/drive/MyDrive/PTQ_ResNet18/models/trt_engine/resnet18_fp32.engine
Saved TensorRT engine: /content/drive/MyDrive/PTQ_ResNet18/models/trt_engine/resnet18_int8_qdq.engine


In [15]:
# Create a held-out validation dataset.
import numpy as np
from datasets import load_dataset

val_ds = load_dataset("AI-Lab-Makerere/beans", split="test")


def make_validation_batch(dataset, transform):
    images = []
    labels = []

    for row in dataset:
        images.append(transform(row["image"].convert("RGB")).numpy().astype(np.float32))
        labels.append(row["labels"])

    return np.stack(images), np.asarray(labels, dtype=np.int64)


x_val, y_val = make_validation_batch(val_ds, transform)
print("Validation inputs:", x_val.shape)
print("Validation labels:", y_val.shape)

Validation inputs: (128, 3, 224, 224)
Validation labels: (128,)


In [16]:
# Test non-TensorRT inference.
def ort_predict(model_path, inputs, providers=("CPUExecutionProvider",)):
    session = ort.InferenceSession(
        str(model_path),
        providers=list(providers),
    )

    input_name = session.get_inputs()[0].name
    expected_batch = session.get_inputs()[0].shape[0]

    def predict_batch(batch):
        return session.run(None, {input_name: batch})[0]

    if isinstance(expected_batch, int) and expected_batch != inputs.shape[0]:
        outputs = [
            predict_batch(inputs[index:index + expected_batch])
            for index in range(0, inputs.shape[0], expected_batch)
        ]
        return np.concatenate(outputs, axis=0)

    return predict_batch(inputs)


fp32_logits = ort_predict(FP32_ONNX, x_val)
ort_int8_logits = ort_predict(INT8_ONNX, x_val)
# ModelOpt writes FP16 tensors into its Q/DQ graph; use CUDA when available.
qdq_logits = ort_predict(
    QDQ_ONNX, x_val, providers=("CUDAExecutionProvider", "CPUExecutionProvider")
)

fp32_predictions = np.argmax(fp32_logits, axis=1)

print("Logit shape:", fp32_logits.shape)
print("Beans labels:", np.unique(y_val))
print("ImageNet prediction range:", fp32_predictions.min(), fp32_predictions.max())
print("Ground-truth Beans accuracy is not reported because this model is not trained on Beans.")

Logit shape: (128, 1000)
Beans labels: [0 1 2]
ImageNet prediction range: 27 990
Ground-truth Beans accuracy is not reported because this model is not trained on Beans.


In [17]:
# Test numerical agreement with FP32.
def compare_outputs(reference, candidate, name):
    absolute_error = np.abs(reference - candidate)
    cosine_similarity = np.sum(reference * candidate, axis=1) / (
        np.linalg.norm(reference, axis=1) *
        np.linalg.norm(candidate, axis=1) + 1e-12
    )
    reference_predictions = np.argmax(reference, axis=1)
    candidate_predictions = np.argmax(candidate, axis=1)
    prediction_agreement = np.mean(reference_predictions == candidate_predictions)

    print(f"\n{name}")
    print(f"Maximum absolute error: {absolute_error.max():.6f}")
    print(f"Mean absolute error: {absolute_error.mean():.6f}")
    print(f"Mean cosine similarity: {cosine_similarity.mean():.6f}")
    print(f"Top-1 prediction agreement: {prediction_agreement:.4f}")
    print("Reference first predictions:", reference_predictions[:10])
    print("Candidate first predictions:", candidate_predictions[:10])
    print("Reference logit range:", reference.min(), reference.max())
    print("Candidate logit range:", candidate.min(), candidate.max())

    return {
        "max_absolute_error": absolute_error.max(),
        "mean_absolute_error": absolute_error.mean(),
        "cosine_similarity": cosine_similarity.mean(),
        "prediction_agreement": prediction_agreement,
    }


def assert_fp32_parity(comparison):
    # These limits deliberately make a wrong-engine/result fail loudly.
    assert comparison["max_absolute_error"] < 1e-3, comparison
    assert comparison["cosine_similarity"] > 0.9999, comparison
    assert comparison["prediction_agreement"] == 1.0, comparison

assert np.isfinite(fp32_logits).all()

ort_int8_comparison = compare_outputs(
    fp32_logits, ort_int8_logits, "ONNX Runtime INT8 vs FP32 ONNX"
)
qdq_comparison = compare_outputs(
    fp32_logits, qdq_logits, "ModelOpt Q/DQ ONNX vs FP32 ONNX"
)

assert ort_int8_logits.shape == fp32_logits.shape
assert qdq_logits.shape == fp32_logits.shape
assert np.isfinite(ort_int8_logits).all()
assert np.isfinite(qdq_logits).all()



ONNX Runtime INT8 vs FP32 ONNX
Maximum absolute error: 6.520893
Mean absolute error: 0.436364
Mean cosine similarity: 0.998389
Top-1 prediction agreement: 0.5703
Reference first predictions: [ 64 988 325  47 325 313 990 325 325 325]
Candidate first predictions: [ 64 988 955  47 325  46 990 325  47 955]
Reference logit range: -17.6894 2.020896
Candidate logit range: -17.12779 2.356204

ModelOpt Q/DQ ONNX vs FP32 ONNX
Maximum absolute error: 1.938836
Mean absolute error: 0.171720
Mean cosine similarity: 0.999799
Top-1 prediction agreement: 0.8281
Reference first predictions: [ 64 988 325  47 325 313 990 325 325 325]
Candidate first predictions: [ 64 988 325  47 325  59 990 325 325 325]
Reference logit range: -17.6894 2.020896
Candidate logit range: -17.171017 2.642563


## Validate the FP32 TensorRT pipeline


In [18]:
# cuda-python is installed in the shared dependency cell above.


In [19]:
import numpy as np
import tensorrt as trt
# CUDA Python 13 moved the runtime API to cuda.bindings.runtime;
# older Colab images expose it as cuda.cudart.
try:
    from cuda.bindings import runtime as cudart
except ImportError:
    from cuda import cudart


def _check_cuda(error_code, operation):
    if error_code != cudart.cudaError_t.cudaSuccess:
        raise RuntimeError(f"{operation} failed: {error_code}")


def trt_predict(engine_path, inputs):
    """Run an engine over `inputs` (N,C,H,W), one image at a time.

    The engine/context/buffers are created ONCE and always released, instead of
    deserializing the engine for every image (which leaked GPU memory).
    Input is cast to the engine's actual input dtype (a strongly-typed Q/DQ
    engine may expect FP16; sending float32 bytes would silently corrupt it).
    """
    if inputs.ndim != 4:
        raise ValueError(f"Expected NCHW image inputs, got shape {inputs.shape}")

    logger = trt.Logger(trt.Logger.ERROR)
    runtime = engine = context = None
    input_device = output_device = stream = None
    try:
        runtime = trt.Runtime(logger)
        engine = runtime.deserialize_cuda_engine(Path(engine_path).read_bytes())
        if engine is None:
            raise RuntimeError("TensorRT could not deserialize the engine")
        context = engine.create_execution_context()

        input_name = output_name = None
        for index in range(engine.num_io_tensors):
            name = engine.get_tensor_name(index)
            if engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT:
                input_name = name
            else:
                output_name = name
        if input_name is None or output_name is None:
            raise RuntimeError("Expected one TensorRT input and one output tensor")

        in_dtype = np.dtype(trt.nptype(engine.get_tensor_dtype(input_name)))
        out_dtype = np.dtype(trt.nptype(engine.get_tensor_dtype(output_name)))
        print(f"{Path(engine_path).name}: input dtype={in_dtype}, output dtype={out_dtype}")

        one_shape = (1,) + tuple(inputs.shape[1:])
        if not context.set_input_shape(input_name, one_shape):
            raise RuntimeError(f"Could not set TensorRT input shape: {one_shape}")
        output_shape = tuple(context.get_tensor_shape(output_name))

        host_in = np.empty(one_shape, dtype=in_dtype)
        host_out = np.empty(output_shape, dtype=out_dtype)

        error, input_device = cudart.cudaMalloc(host_in.nbytes)
        _check_cuda(error, "cudaMalloc(input)")
        error, output_device = cudart.cudaMalloc(host_out.nbytes)
        _check_cuda(error, "cudaMalloc(output)")
        error, stream = cudart.cudaStreamCreate()
        _check_cuda(error, "cudaStreamCreate")

        context.set_tensor_address(input_name, int(input_device))
        context.set_tensor_address(output_name, int(output_device))

        results = []
        for i in range(len(inputs)):
            host_in[...] = inputs[i:i + 1]  # casts to the engine dtype
            _check_cuda(cudart.cudaMemcpyAsync(
                input_device, host_in.ctypes.data, host_in.nbytes,
                cudart.cudaMemcpyKind.cudaMemcpyHostToDevice, stream)[0],
                "cudaMemcpyAsync(input)")
            if not context.execute_async_v3(stream):
                raise RuntimeError("TensorRT execution failed")
            _check_cuda(cudart.cudaMemcpyAsync(
                host_out.ctypes.data, output_device, host_out.nbytes,
                cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost, stream)[0],
                "cudaMemcpyAsync(output)")
            _check_cuda(cudart.cudaStreamSynchronize(stream)[0], "cudaStreamSynchronize")
            results.append(host_out.astype(np.float32))
        return np.concatenate(results, axis=0)
    finally:
        if stream is not None:
            cudart.cudaStreamDestroy(stream)
        if input_device is not None:
            cudart.cudaFree(input_device)
        if output_device is not None:
            cudart.cudaFree(output_device)
        del context, engine, runtime


In [20]:
require_gpu_runtime()

trt_fp32_logits = trt_predict(FP32_ENGINE_PATH, x_val)
trt_int8_logits = trt_predict(INT8_ENGINE_PATH, x_val)
trt_fp32_predictions = np.argmax(trt_fp32_logits, axis=1)

print("FP32 TensorRT logit shape:", trt_fp32_logits.shape)
print("INT8 TensorRT logit shape:", trt_int8_logits.shape)

trt_fp32_comparison = compare_outputs(
    fp32_logits,
    trt_fp32_logits,
    "FP32 TensorRT vs FP32 ONNX",
)

assert trt_fp32_logits.shape == fp32_logits.shape
assert np.isfinite(trt_fp32_logits).all()
assert_fp32_parity(trt_fp32_comparison)
print("FP32 TensorRT parity check passed.")

trt_int8_vs_qdq_comparison = compare_outputs(
    qdq_logits, trt_int8_logits, "TensorRT INT8 vs ModelOpt Q/DQ ONNX"
)
trt_int8_vs_fp32_comparison = compare_outputs(
    fp32_logits, trt_int8_logits, "TensorRT INT8 vs FP32 ONNX"
)

assert trt_int8_logits.shape == fp32_logits.shape
assert np.isfinite(trt_int8_logits).all()

# INT8 quality guard: finite values are not enough. Warn loudly when the
# quantized model has clearly diverged from FP32.
for name, cmp in [("ORT INT8", ort_int8_comparison),
                  ("ModelOpt Q/DQ", qdq_comparison),
                  ("TensorRT INT8", trt_int8_vs_fp32_comparison)]:
    if cmp["cosine_similarity"] < 0.98:
        print(f"WARNING: {name} diverges from FP32 "
              f"(cosine={cmp['cosine_similarity']:.3f}); treat its results as BROKEN.")


resnet18_fp32.engine: input dtype=float32, output dtype=float32
resnet18_int8_qdq.engine: input dtype=float32, output dtype=float32
FP32 TensorRT logit shape: (128, 1000)
INT8 TensorRT logit shape: (128, 1000)

FP32 TensorRT vs FP32 ONNX
Maximum absolute error: 0.000011
Mean absolute error: 0.000002
Mean cosine similarity: 1.000000
Top-1 prediction agreement: 1.0000
Reference first predictions: [ 64 988 325  47 325 313 990 325 325 325]
Candidate first predictions: [ 64 988 325  47 325 313 990 325 325 325]
Reference logit range: -17.6894 2.020896
Candidate logit range: -17.6894 2.0209
FP32 TensorRT parity check passed.

TensorRT INT8 vs ModelOpt Q/DQ ONNX
Maximum absolute error: 0.712571
Mean absolute error: 0.079788
Mean cosine similarity: 0.999957
Top-1 prediction agreement: 0.9219
Reference first predictions: [ 64 988 325  47 325  59 990 325 325 325]
Candidate first predictions: [ 64 988 325  47 325  46 990 325 325 325]
Reference logit range: -17.171017 2.642563
Candidate logit range

In [21]:
# Latency / size benchmark, following the paper's FP32-vs-INT8 comparison.
# Engines are built with static batch 1; timing excludes host<->device copies.
import time
import numpy as np
import torch
import tensorrt as trt


def trt_latency(engine_path, warmup=50, iters=300):
    logger = trt.Logger(trt.Logger.ERROR)
    runtime = trt.Runtime(logger)
    engine = runtime.deserialize_cuda_engine(Path(engine_path).read_bytes())
    context = engine.create_execution_context()
    names = [engine.get_tensor_name(i) for i in range(engine.num_io_tensors)]
    in_name = next(n for n in names if engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT)
    out_name = next(n for n in names if n != in_name)
    to_torch = {np.dtype(np.float32): torch.float32, np.dtype(np.float16): torch.float16}
    in_dt = to_torch[np.dtype(trt.nptype(engine.get_tensor_dtype(in_name)))]
    out_dt = to_torch[np.dtype(trt.nptype(engine.get_tensor_dtype(out_name)))]
    shape = (1,) + tuple(x_val.shape[1:])
    context.set_input_shape(in_name, shape)
    x = torch.from_numpy(x_val[:1]).to("cuda", in_dt).contiguous()
    y = torch.empty(tuple(context.get_tensor_shape(out_name)), dtype=out_dt, device="cuda")
    context.set_tensor_address(in_name, x.data_ptr())
    context.set_tensor_address(out_name, y.data_ptr())
    stream = torch.cuda.Stream()
    times = []
    with torch.cuda.stream(stream):
        for i in range(warmup + iters):
            start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            start.record(stream)
            context.execute_async_v3(stream.cuda_stream)
            end.record(stream)
            stream.synchronize()
            if i >= warmup:
                times.append(start.elapsed_time(end))
    del context, engine, runtime
    return float(np.mean(times)), float(np.percentile(times, 95))


def ort_latency(model_path, warmup=10, iters=100):
    session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    name = session.get_inputs()[0].name
    x = x_val[:1]
    for _ in range(warmup):
        session.run(None, {name: x})
    times = []
    for _ in range(iters):
        t0 = time.perf_counter()
        session.run(None, {name: x})
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.percentile(times, 95))


require_gpu_runtime()
mb = lambda path: Path(path).stat().st_size / 1e6
latency = {
    "fp32_trt": trt_latency(FP32_ENGINE_PATH),
    "int8_trt": trt_latency(INT8_ENGINE_PATH),
    "fp32_ort": ort_latency(FP32_ONNX),
    "int8_ort": ort_latency(INT8_ONNX),
}
for k, (mean_ms, p95_ms) in latency.items():
    print(f"{k}: mean {mean_ms:.3f} ms, p95 {p95_ms:.3f} ms")
print(f"TensorRT speedup (FP32/INT8): {latency['fp32_trt'][0] / latency['int8_trt'][0]:.2f}x")
print(f"Engine size: FP32 {mb(FP32_ENGINE_PATH):.2f} MB -> INT8 {mb(INT8_ENGINE_PATH):.2f} MB "
      f"({mb(FP32_ENGINE_PATH) / mb(INT8_ENGINE_PATH):.2f}x smaller)")


fp32_trt: mean 1.349 ms, p95 1.365 ms
int8_trt: mean 0.417 ms, p95 0.427 ms
fp32_ort: mean 32.770 ms, p95 38.867 ms
int8_ort: mean 21.842 ms, p95 24.684 ms
TensorRT speedup (FP32/INT8): 3.23x
Engine size: FP32 57.68 MB -> INT8 13.85 MB (4.16x smaller)


In [24]:
import pandas as pd


def model_mb(path):
    """ONNX size including external weight files (<name>.onnx.data / .onnx_data)."""
    path = Path(path)
    total = path.stat().st_size
    for extra in path.parent.glob(path.name + "*"):
        if extra != path:
            total += extra.stat().st_size
    return total / 1e6


results = pd.DataFrame([
    {"pipeline": "FP32 ONNX Runtime (CPU)", "size_mb": model_mb(FP32_ONNX),
     "latency_ms": latency["fp32_ort"][0], "cosine_vs_fp32": 1.0, "top1_agreement": 1.0},
    {"pipeline": "ONNX Runtime INT8 (CPU)", "size_mb": model_mb(INT8_ONNX),
     "latency_ms": latency["int8_ort"][0], "cosine_vs_fp32": ort_int8_comparison["cosine_similarity"],
     "top1_agreement": ort_int8_comparison["prediction_agreement"]},
    {"pipeline": "FP32 TensorRT (TF32 off)", "size_mb": FP32_ENGINE_PATH.stat().st_size / 1e6,
     "latency_ms": latency["fp32_trt"][0], "cosine_vs_fp32": trt_fp32_comparison["cosine_similarity"],
     "top1_agreement": trt_fp32_comparison["prediction_agreement"]},
    {"pipeline": "INT8 TensorRT (ModelOpt Q/DQ)", "size_mb": INT8_ENGINE_PATH.stat().st_size / 1e6,
     "latency_ms": latency["int8_trt"][0], "cosine_vs_fp32": trt_int8_vs_fp32_comparison["cosine_similarity"],
     "top1_agreement": trt_int8_vs_fp32_comparison["prediction_agreement"]},
]).round(4)

# Accuracy here means agreement with the FP32 model (no labels: the model is an
# ImageNet classifier and Beans is only used as an input distribution).
results


,pipeline,size_mb,latency_ms,cosine_vs_fp32,top1_agreement
0,FP32 ONNX Runtime (CPU),46.8804,32.7704,1.0000,1.0000
1,ONNX Runtime INT8 (CPU),11.8670,21.8421,0.9984,0.5703
2,FP32 TensorRT (TF32 off),57.6808,1.3488,1.0000,1.0000
3,INT8 TensorRT (ModelOpt Q/DQ),13.8513,0.4170,0.9998,0.8203
